In [1]:
import re
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from clang import cindex

# Set libclang path (update this if needed)
cindex.Config.set_library_file(r"C:\Program Files\LLVM\bin\libclang.dll")

device = torch.device('cpu')

# Initialize Clang Index
index = cindex.Index.create()

def generate_ast_clang(source_code, temp_file_path="temp.c"):
    """Generate AST from source code using libclang."""
    try:
        with open(temp_file_path, "w") as temp_file:
            temp_file.write(source_code)
        translation_unit = index.parse(temp_file_path, args=['-std=c99'])
        return translation_unit.cursor
    except Exception as e:
        print(f"Error during parsing: {e}")
        return None

def generate_ast_string(node, depth=0):
    """Convert the AST node to a readable indented string."""
    indent = "  " * depth
    result = f"{indent}{node.kind} {node.spelling}\n"
    for child in node.get_children():
        result += generate_ast_string(child, depth + 1)
    return result

def generate_and_parse_ast_string(source_code):
    """Generate the AST string from C code."""
    try:
        root = generate_ast_clang(source_code)
        if root is not None:
            return generate_ast_string(root)
        else:
            return "Error"
    except Exception as e:
        return f"Error: {e}"

def parse_ast_to_graph(ast_list, label, node_encoder):
    """Convert AST list into a torch_geometric graph."""
    node_ids = [node_encoder.get(node, 0) for node in ast_list]

    x = F.one_hot(torch.tensor(node_ids), num_classes=len(node_encoder)).float()

    edge_index = []
    for i in range(len(node_ids) - 1):
        edge_index.append([i, i + 1])
        edge_index.append([i + 1, i])  # bidirectional

    edge_index = torch.tensor(edge_index).t().contiguous()
    y = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, y=y)

def load_dataset_chunk(csv_path, start, end, node_encoder=None):
    """Load a chunk of dataset and convert to graphs."""
    df = pd.read_excel(csv_path)
    df = df.drop(columns=["commit_id", "project"])
    df = df.iloc[start:end].reset_index(drop=True)

    print("Generating ASTs and collecting node types...")
    ast_strings = []
    ast_lists = []
    # all_node_types = set()

    for code in tqdm(df['func']):
        try:
            # cleaned = wrap_c_code(clean_c_code(code))
            ast_string = generate_and_parse_ast_string(code)
            ast_strings.append(ast_string)

            # if ast_string.startswith("Error"):
            #     ast_list = []
            # else:
            #     ast_list = [line.strip().split()[0] for line in ast_string.strip().split("\n") if line.strip()]
            # ast_lists.append(ast_list)
            # all_node_types.update(ast_list)
        except Exception as e:
            ast_strings.append("Error")
            ast_lists.append([])

    df['AST'] = ast_strings
    df['AST_list'] = df['AST'].apply(lambda s: s.strip().split())
    # Initialize encoder if needed
    all_nodes = set(node for row in df['AST_list'] for node in row)
    # node_encoder = {node: idx for idx, node in enumerate(sorted(all_nodes))}
    # torch.save(node_encoder, "node_encoder.pt")
    graphs = []
    for ast, label in tqdm(zip(df['AST_list'], df['target']), total=len(df)):
        if ast:
            data = parse_ast_to_graph(ast, label, node_encoder)
            graphs.append(data)

    return graphs, node_encoder

class GNN(torch.nn.Module):
    """Graph Neural Network using SAGEConv."""
    def __init__(self, input_dim, output_dim, dropout=0.3):
        super(GNN, self).__init__()
        self.conv1 = SAGEConv(input_dim, 256)
        self.conv2 = SAGEConv(256, 128)
        self.conv3 = SAGEConv(128, 64)
        self.lin = torch.nn.Linear(64, output_dim)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)

        return F.log_softmax(self.lin(x), dim=1)

def calculate_accuracy(pred, labels):
    """Calculate accuracy from predictions and true labels."""
    pred_labels = pred.max(dim=1)[1]  # Get predicted class
    correct = (pred_labels == labels).sum().item()
    return correct / len(labels)

def train_with_early_stopping(model, loader, optimizer, device, patience=2, max_epochs=5):
    """Train model with early stopping and accuracy logging."""
    best_loss = float('inf')
    best_model = None
    trigger_times = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        for data in loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = F.nll_loss(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # ✅ Accuracy calculation
            acc = calculate_accuracy(out, data.y)
            total_correct += acc * data.y.size(0)
            total_samples += data.y.size(0)

        avg_loss = total_loss / len(loader)
        avg_acc = total_correct / total_samples

        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model = model.state_dict()
            trigger_times = 0
        else:
            trigger_times += 1
            print(f"Early stopping counter: {trigger_times}/{patience}")
            if trigger_times >= patience:
                print("Early stopping triggered.")
                break

    if best_model:
        model.load_state_dict(best_model)
    return model


c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csv_path = "output.xlsx"
start, end = 0, 3000  # First chunk

node_encoder = torch.load("node_encoder.pt")
graphs, _ = load_dataset_chunk(csv_path, start, end,node_encoder=node_encoder)

# DataLoader
train_loader = DataLoader(graphs, batch_size=16, shuffle=True)

# Model
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training
model = train_with_early_stopping(model, train_loader, optimizer, device)

# Save
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_77888\2920176233.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:50<00:00, 59.64it/s] 


Epoch 1 | Loss: 0.6907 | Accuracy: 0.5280
Epoch 2 | Loss: 0.6172 | Accuracy: 0.6583
Epoch 3 | Loss: 0.3228 | Accuracy: 0.8650
Epoch 4 | Loss: 0.1360 | Accuracy: 0.9523
Epoch 5 | Loss: 0.0929 | Accuracy: 0.9710
Epoch 6 | Loss: 0.0711 | Accuracy: 0.9727
Epoch 7 | Loss: 0.0551 | Accuracy: 0.9760
Epoch 8 | Loss: 0.0482 | Accuracy: 0.9793
Epoch 9 | Loss: 0.0421 | Accuracy: 0.9793
Epoch 10 | Loss: 0.0399 | Accuracy: 0.9787


In [2]:


node_encoder = torch.load("node_encoder.pt")
graphs2, _ = load_dataset_chunk("output.xlsx", 3000,6000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader2 = DataLoader(graphs2, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader2, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_21492\2529130436.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [01:00<00:00, 49.89it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_21492\2529130436.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.

Epoch 1 | Loss: 0.7847 | Accuracy: 0.5287
Epoch 2 | Loss: 0.6682 | Accuracy: 0.5793
Epoch 3 | Loss: 0.5456 | Accuracy: 0.7223
Epoch 4 | Loss: 0.2750 | Accuracy: 0.8877
Epoch 5 | Loss: 0.1605 | Accuracy: 0.9430
Epoch 6 | Loss: 0.1071 | Accuracy: 0.9617
Epoch 7 | Loss: 0.0860 | Accuracy: 0.9673
Epoch 8 | Loss: 0.0751 | Accuracy: 0.9680
Epoch 9 | Loss: 0.0658 | Accuracy: 0.9743
Epoch 10 | Loss: 0.0622 | Accuracy: 0.9740


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs3, _ = load_dataset_chunk("output.xlsx", 6000,9000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader3 = DataLoader(graphs3, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader3, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_22172\1906239092.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:51<00:00, 58.76it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_22172\1906239092.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.

Epoch 1 | Loss: 0.8166 | Accuracy: 0.5320
Epoch 2 | Loss: 0.6269 | Accuracy: 0.6480
Epoch 3 | Loss: 0.4638 | Accuracy: 0.7863
Epoch 4 | Loss: 0.2751 | Accuracy: 0.8933
Epoch 5 | Loss: 0.1523 | Accuracy: 0.9413
Epoch 6 | Loss: 0.1078 | Accuracy: 0.9560
Epoch 7 | Loss: 0.0829 | Accuracy: 0.9650
Epoch 8 | Loss: 0.0786 | Accuracy: 0.9680
Epoch 9 | Loss: 0.0670 | Accuracy: 0.9717
Epoch 10 | Loss: 0.0588 | Accuracy: 0.9697


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs4, _ = load_dataset_chunk("output.xlsx", 9000,12000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader4 = DataLoader(graphs4, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader4, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_92948\559159172.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:54<00:00, 55.41it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_92948\559159172.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.l

Epoch 1 | Loss: 0.8024 | Accuracy: 0.5250
Epoch 2 | Loss: 0.6280 | Accuracy: 0.6513
Epoch 3 | Loss: 0.4971 | Accuracy: 0.7733
Epoch 4 | Loss: 0.2703 | Accuracy: 0.8903
Epoch 5 | Loss: 0.1505 | Accuracy: 0.9423
Epoch 6 | Loss: 0.0961 | Accuracy: 0.9623
Epoch 7 | Loss: 0.0720 | Accuracy: 0.9717
Epoch 8 | Loss: 0.0637 | Accuracy: 0.9710
Epoch 9 | Loss: 0.0538 | Accuracy: 0.9740
Epoch 10 | Loss: 0.0462 | Accuracy: 0.9763


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs5, _ = load_dataset_chunk("output.xlsx", 12000,15000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader5 = DataLoader(graphs5, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader5, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_8096\1945449218.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:40<00:00, 74.67it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_8096\1945449218.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.l

Epoch 1 | Loss: 0.8096 | Accuracy: 0.5253
Epoch 2 | Loss: 0.6521 | Accuracy: 0.5973
Epoch 3 | Loss: 0.5916 | Accuracy: 0.6633
Epoch 4 | Loss: 0.4669 | Accuracy: 0.7613
Epoch 5 | Loss: 0.2702 | Accuracy: 0.8957


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs6, _ = load_dataset_chunk("output.xlsx", 15000,18000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader6 = DataLoader(graphs6, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader6, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_45440\2265420296.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:52<00:00, 57.24it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_45440\2265420296.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.

Epoch 1 | Loss: 0.7638 | Accuracy: 0.5557
Epoch 2 | Loss: 0.5851 | Accuracy: 0.6977
Epoch 3 | Loss: 0.4245 | Accuracy: 0.8163
Epoch 4 | Loss: 0.2456 | Accuracy: 0.9050
Epoch 5 | Loss: 0.1509 | Accuracy: 0.9450


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs7, _ = load_dataset_chunk("output.xlsx", 18000,21000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader7 = DataLoader(graphs7, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader7, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_14320\2794890613.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:51<00:00, 58.15it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_14320\2794890613.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.

Epoch 1 | Loss: 0.7997 | Accuracy: 0.5417
Epoch 2 | Loss: 0.5798 | Accuracy: 0.6967
Epoch 3 | Loss: 0.4086 | Accuracy: 0.8257
Epoch 4 | Loss: 0.2447 | Accuracy: 0.9070
Epoch 5 | Loss: 0.1428 | Accuracy: 0.9463


In [2]:
node_encoder = torch.load("node_encoder.pt")
graphs8, _ = load_dataset_chunk("output.xlsx", 21000,24000, node_encoder=node_encoder)


# DataLoader for second chunk
train_loader8 = DataLoader(graphs8, batch_size=16, shuffle=True)

# Load the model and encoder
model = GNN(input_dim=len(node_encoder), output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train on new chunk
model = train_with_early_stopping(model, train_loader8, optimizer, device)

# Save again
torch.save(model.state_dict(), "gnn_trained.pth")

C:\Users\ramim\AppData\Local\Temp\ipykernel_12804\2211146071.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3000/3000 [00:53<00:00, 55.81it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_12804\2211146071.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.

Epoch 1 | Loss: 0.8002 | Accuracy: 0.5577
Epoch 2 | Loss: 0.5871 | Accuracy: 0.6890
Epoch 3 | Loss: 0.4264 | Accuracy: 0.8167
Epoch 4 | Loss: 0.2399 | Accuracy: 0.9063
Epoch 5 | Loss: 0.1292 | Accuracy: 0.9540


In [2]:
from sklearn.metrics import accuracy_score, classification_report

@torch.no_grad()
def test_model(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1).cpu().numpy()
        label = data.y.cpu().numpy()

        all_preds.extend(pred)
        all_labels.extend(label)

    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=["Not Vulnerable", "Vulnerable"])
    print(f"\nTest Accuracy: {acc:.4f}")
    print("\nClassification Report:\n", report)


In [3]:
# Load final chunk
start, end = 24000, 27318
node_encoder = torch.load("node_encoder.pt")
test_graphs, _ = load_dataset_chunk("output.xlsx", start, end, node_encoder=node_encoder)

# DataLoader for test
test_loader = DataLoader(test_graphs, batch_size=16)

# Load model
model = GNN(input_dim=len(node_encoder),output_dim=2, dropout=0.3).to(device)
model.load_state_dict(torch.load("gnn_trained.pth"))

# Test
test_model(model, test_loader, device)


C:\Users\ramim\AppData\Local\Temp\ipykernel_28192\382334736.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node_encoder = torch.load("node_encoder.pt")


Generating ASTs and collecting node types...


100%|██████████| 3318/3318 [00:59<00:00, 55.62it/s] 
C:\Users\ramim\AppData\Local\Temp\ipykernel_28192\382334736.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.l


Test Accuracy: 0.9301

Classification Report:
                 precision    recall  f1-score   support

Not Vulnerable       0.96      0.90      0.93      1759
    Vulnerable       0.90      0.96      0.93      1559

      accuracy                           0.93      3318
     macro avg       0.93      0.93      0.93      3318
  weighted avg       0.93      0.93      0.93      3318

